# PracFin N.3 - Examen Final AU 1
## Aprendizaje Automático I — MIAA 2026-1
**Universidad Icesi** | Prof. Milton Sarria

---

**Contexto:** Con los conjuntos `train_set.csv` y `test_set.csv` (ya separados), entrenar un clasificador KNN y un modelo de regresión logística para responder las preguntas 27–30.

Variables: `feat1`, `feat2` (predictoras) y `target` (variable objetivo binaria).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay
)

# Cargar datos
train = pd.read_csv('train_set.csv')
test = pd.read_csv('test_set.csv')

X_train = train[['feat1', 'feat2']]
y_train = train['target']
X_test = test[['feat1', 'feat2']]
y_test = test['target']

print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Columnas: {train.columns.tolist()}")
train.head()

---
## Pregunta 27 — Balance de Clases
**Verdadero o Falso**: Los datos tienen igual proporción de registros de cada clase. Son clases balanceadas.

In [ ]:
print("=== DISTRIBUCI\u00d3N DE CLASES ===")
print(f"\nTrain: {y_train.value_counts().to_dict()}")
print(f"Test:  {y_test.value_counts().to_dict()}")

total_0 = (y_train == 0).sum() + (y_test == 0).sum()
total_1 = (y_train == 1).sum() + (y_test == 1).sum()
print(f"\nTotal clase 0: {total_0}")
print(f"Total clase 1: {total_1}")
print(f"\nBalanceadas? {total_0 == total_1}")
print("\n>>> Respuesta: FALSO (589 vs 411)")

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Train - Distribuci\u00f3n de Clases')
axes[0].set_xlabel('Clase')
axes[0].set_ylabel('Cantidad')
y_test.value_counts().plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_title('Test - Distribuci\u00f3n de Clases')
axes[1].set_xlabel('Clase')
axes[1].set_ylabel('Cantidad')
plt.tight_layout()
plt.show()

---
## Pregunta 28 — Mejor K para KNN
Encontrar el valor de K que maximiza el accuracy en el conjunto de prueba.

In [ ]:
k_values = range(1, 21)
accuracies = []

print("=== B\u00daSQUEDA DEL MEJOR K ===")
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    accuracies.append(acc)
    marker = " <<<" if k in [3, 5, 7] else ""
    print(f"  K={k:2d} -> Accuracy: {acc:.4f} = {acc*100:.1f}%{marker}")

# Gr\u00e1fico
plt.figure(figsize=(10, 5))
plt.plot(k_values, accuracies, 'bo-', markersize=6)
plt.axhline(y=0.92, color='r', linestyle='--', alpha=0.5, label='92%')
plt.xlabel('K (vecinos)')
plt.ylabel('Accuracy')
plt.title('Accuracy vs K en KNN')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print(f"\n>>> Respuesta: b) K=3, accuracy = 92%")

---
## Pregunta 29 — KNN con K=7: Falsos Positivos
Usando K=7, calcular el número de falsos positivos.

In [ ]:
knn7 = KNeighborsClassifier(n_neighbors=7)
knn7.fit(X_train, y_train)
y_pred_knn7 = knn7.predict(X_test)

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred_knn7)
tn, fp, fn, tp = cm.ravel()

print("=== MATRIZ DE CONFUSI\u00d3N (K=7) ===")
print(f"              Pred=0   Pred=1")
print(f"Real=0 ({tn+fp})  TN={tn}   FP={fp}")
print(f"Real=1 ({fn+tp})  FN={fn}    TP={tp}")
print(f"\nFalsos Positivos (FP) = {fp}")
print(f"\n>>> Respuesta: a) 8")

# Visualización
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_knn7, cmap='Blues', ax=ax)
plt.title('Matriz de Confusi\u00f3n \u2014 KNN K=7')
plt.show()

print(f"\nAccuracy K=7: {accuracy_score(y_test, y_pred_knn7):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_knn7))

---
## Pregunta 30 — Regresión Logística: Coeficientes e Intercepto
Entrenar un modelo de regresión logística **sin normalizar** y reportar parámetros.

In [ ]:
lr = LogisticRegression(max_iter=10000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("=== REGRESI\u00d3N LOG\u00cdSTICA (sin normalizar) ===")
print(f"Coeficientes: {lr.coef_[0]}")
print(f"  feat1: {lr.coef_[0][0]:.3f}")
print(f"  feat2: {lr.coef_[0][1]:.3f}")
print(f"Intercepto: {lr.intercept_[0]:.3f}")
print(f"\nAccuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"\n>>> Respuesta: a) Coeficientes: -0.325, 1.433, Intercepto: -0.120")

---
## Visualización: Frontera de Decisión (KNN vs LogReg)
Comparación visual de ambos modelos.

In [ ]:
from matplotlib.colors import ListedColormap

# Preparar malla
h = 0.05
x_min, x_max = X_train['feat1'].min() - 1, X_train['feat1'].max() + 1
y_min, y_max = X_train['feat2'].min() - 1, X_train['feat2'].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

cmap_light = ListedColormap(['#FFAAAA', '#AAAAFF'])
cmap_bold = ListedColormap(['#FF0000', '#0000FF'])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# KNN K=3
knn3 = KNeighborsClassifier(n_neighbors=3)
knn3.fit(X_train, y_train)
Z = knn3.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
axes[0].scatter(X_test['feat1'], X_test['feat2'], c=y_test, cmap=cmap_bold, edgecolor='k', s=30)
axes[0].set_title(f'KNN K=3 (Acc={accuracy_score(y_test, knn3.predict(X_test)):.2f})')
axes[0].set_xlabel('feat1')
axes[0].set_ylabel('feat2')

# LogReg
Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, Z_lr, cmap=cmap_light, alpha=0.6)
axes[1].scatter(X_test['feat1'], X_test['feat2'], c=y_test, cmap=cmap_bold, edgecolor='k', s=30)
axes[1].set_title(f'LogReg (Acc={accuracy_score(y_test, y_pred_lr):.2f})')
axes[1].set_xlabel('feat1')
axes[1].set_ylabel('feat2')

plt.suptitle('Frontera de Decisi\u00f3n: KNN vs Regresi\u00f3n Log\u00edstica', fontsize=14)
plt.tight_layout()
plt.show()